# Refined QAD differentiation with proliferation

This notebook provides a simple example of neuronal stem-cell differentiation. 
The latent states are `Q1` (deep quiescent), `Q2` (shallow quiescent), `A` (activated), `M1` (early mitotic), `M2` (late mitotic, proliferating), and `D` (differentiated).
Cells differentiate through the graph, only `M2` divides, and the accumulation of terminal `D` cells slows both transitions and division through a Hill feedback term.

The configuration starts with 100 cells distributed over `Q1` and `Q2` in a 40-60 ratio, and expands to about 1000 cells.

## Preparations

In [ ]:
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from markovmodus import SimulationParameters, SimulationState, simulate_dataset

In [ ]:
# Helper functions.
def num_states_from_adata(adata) -> int:
    return int(np.asarray(adata.uns["state_expression"]).shape[0])


def state_palette(num_states: int) -> list:
    cmap = plt.get_cmap("tab10")
    return [cmap(state % cmap.N) for state in range(num_states)]


def plot_transition_graph(
    matrix: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    positions: np.ndarray | None = None,
    palette: list | None = None,
    state_names: list[str] | None = None,
    division_source: int | None = None,
    division_rate: float = 0.0,
    daughter_state_probs: np.ndarray | None = None,
) -> None:
    matrix = np.asarray(matrix, dtype=float)
    num_states = matrix.shape[0]
    if positions is None:
        angles = np.linspace(0, 2 * np.pi, num_states, endpoint=False)
        positions = np.stack((np.cos(angles), np.sin(angles)), axis=1)
    else:
        positions = np.asarray(positions, dtype=float)

    max_rate = float(matrix.max())
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_aspect("equal")
    margin = 0.45
    ax.set_xlim(float(positions[:, 0].min() - margin), float(positions[:, 0].max() + margin))
    ax.set_ylim(float(positions[:, 1].min() - margin), float(positions[:, 1].max() + margin))
    ax.axis("off")

    node_radius = 0.12
    if palette is None:
        palette = state_palette(num_states)
    if state_names is None:
        state_names = [str(state) for state in range(num_states)]
    edge_color = "0.58"

    if max_rate > 0:
        for source in range(num_states):
            for target in range(num_states):
                rate = matrix[source, target]
                if source == target or rate <= 0:
                    continue

                source_pos = positions[source]
                target_pos = positions[target]
                bidirectional = matrix[target, source] > 0
                rad = 0.20 if bidirectional else 0.03
                linewidth = 1.4 + 1.8 * (rate / max_rate)

                ax.annotate(
                    "",
                    xy=target_pos,
                    xytext=source_pos,
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=edge_color,
                        alpha=0.85,
                        lw=linewidth,
                        mutation_scale=14,
                        shrinkA=18,
                        shrinkB=18,
                        connectionstyle=f"arc3,rad={rad}",
                    ),
                    zorder=1,
                )

                direction = target_pos - source_pos
                normal = np.array([-direction[1], direction[0]])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                label_position = 0.55 * source_pos + 0.45 * target_pos + normal * rad * 0.55
                ax.text(
                    label_position[0],
                    label_position[1],
                    f"{rate:.2f}",
                    fontsize=8,
                    color="0.35",
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.8),
                    zorder=2,
                )

    if division_source is not None and daughter_state_probs is not None and division_rate > 0:
        daughter_state_probs = np.asarray(daughter_state_probs, dtype=float)
        row = daughter_state_probs[division_source].copy()
        row_sum = row.sum()
        if row_sum > 0:
            row = row / row_sum
            division_edges = 2.0 * division_rate * row
            max_division_edge = float(division_edges.max())
            source_pos = positions[division_source]
            division_color = palette[division_source % len(palette)]
            for target, edge_rate in enumerate(division_edges):
                if edge_rate <= 0:
                    continue

                target_pos = positions[target]
                linewidth = 1.2 + 2.5 * edge_rate / max_division_edge
                curve = 0.12 if target >= division_source else -0.18
                if target == division_source:
                    curve = 0.35
                ax.annotate(
                    "",
                    xy=target_pos,
                    xytext=source_pos,
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=division_color,
                        linestyle="--",
                        alpha=0.82,
                        lw=linewidth,
                        mutation_scale=14,
                        shrinkA=22,
                        shrinkB=22,
                        connectionstyle=f"arc3,rad={curve}",
                    ),
                    zorder=2,
                )

                direction = target_pos - source_pos
                normal = np.array([-direction[1], direction[0]])
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                if normal[1] < 0:
                    normal = -normal
                label_position = 0.58 * source_pos + 0.42 * target_pos + normal * 0.16
                ax.text(
                    label_position[0],
                    label_position[1],
                    f"{edge_rate:.2f}",
                    fontsize=8,
                    color=division_color,
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=0.8),
                    zorder=3,
                )

            ax.legend(
                handles=[
                    Line2D([0], [0], color=edge_color, lw=2.0, label="transition rate"),
                    Line2D(
                        [0],
                        [0],
                        color=division_color,
                        lw=2.0,
                        linestyle="--",
                        label="2 x division rate x fate probability",
                    ),
                ],
                loc="lower left",
                frameon=False,
                fontsize=7,
            )

    for state, (x, y) in enumerate(positions):
        node = plt.Circle(
            (x, y),
            node_radius,
            facecolor=palette[state % len(palette)],
            edgecolor="black",
            linewidth=1.2,
            alpha=0.85,
            zorder=3,
        )
        ax.add_patch(node)
        ax.text(
            x,
            y,
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=11,
            weight="bold",
            zorder=4,
        )


def preprocess_for_scanpy(adata):
    adata.obs["state"] = adata.obs["state"].astype("category")
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    adata.layers["log_normalized"] = adata.X.copy()

    analysis = adata.copy()
    sc.pp.scale(analysis, max_value=10)
    n_comps = min(20, analysis.n_obs - 1, analysis.n_vars - 1)
    sc.tl.pca(analysis, n_comps=n_comps)
    sc.pp.neighbors(analysis, n_neighbors=min(15, analysis.n_obs - 1))
    sc.tl.umap(analysis)
    adata.obsm["X_pca"] = analysis.obsm["X_pca"].copy()
    adata.obsm["X_umap"] = analysis.obsm["X_umap"].copy()
    return adata


def state_counts(adata) -> pd.Series:
    counts = adata.obs["state"].astype(int).value_counts().reindex(
        range(num_states_from_adata(adata)), fill_value=0
    )
    counts.index.name = "state"
    return counts


def plot_state_counts(
    adata,
    title: str,
    *,
    ax: plt.Axes,
    palette: list | None = None,
    state_names: list[str] | None = None,
) -> None:
    counts = state_counts(adata)
    if palette is None:
        palette = state_palette(num_states_from_adata(adata))
    if state_names is None:
        state_names = [str(state) for state in counts.index]
    bar_colors = [palette[int(state) % len(palette)] for state in counts.index]
    counts.plot(kind="bar", ax=ax, color=bar_colors)
    ax.set_xlabel("State")
    ax.set_ylabel("Cells")
    ax.set_xticklabels(state_names, rotation=0)
    ax.set_title(title)


def plot_embedding_with_state_labels(
    adata,
    coordinates: np.ndarray,
    title: str,
    *,
    ax: plt.Axes,
    palette: list,
    xlabel: str,
    ylabel: str,
    state_names: list[str] | None = None,
) -> None:
    states = adata.obs["state"].astype(int).to_numpy()
    if state_names is None:
        state_names = [str(state) for state in range(num_states_from_adata(adata))]
    for state in range(num_states_from_adata(adata)):
        mask = states == state
        if not mask.any():
            continue
        color = palette[state % len(palette)]
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=18,
            alpha=1.0,
            color=color,
            linewidths=0,
            zorder=1,
        )
        center = coordinates[mask].mean(axis=0)
        ax.scatter(
            center[0],
            center[1],
            s=520,
            color=color,
            edgecolor="black",
            linewidth=1.2,
            alpha=0.7,
            zorder=3,
        )
        ax.text(
            center[0],
            center[1],
            state_names[state],
            ha="center",
            va="center",
            color="white",
            fontsize=10,
            weight="bold",
            zorder=4,
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)


def plot_scanpy_views(
    adata,
    title_prefix: str,
    *,
    axes: tuple[plt.Axes, plt.Axes],
    palette: list,
    state_names: list[str] | None = None,
) -> None:
    plot_embedding_with_state_labels(
        adata,
        adata.obsm["X_pca"][:, :2],
        f"{title_prefix}: PCA",
        ax=axes[0],
        palette=palette,
        xlabel="PC1",
        ylabel="PC2",
        state_names=state_names,
    )
    plot_embedding_with_state_labels(
        adata,
        adata.obsm["X_umap"],
        f"{title_prefix}: UMAP",
        ax=axes[1],
        palette=palette,
        xlabel="UMAP1",
        ylabel="UMAP2",
        state_names=state_names,
    )


def _gene_indices(var_names, genes: list[str]) -> list[int]:
    return [int(np.where(var_names == gene)[0][0]) for gene in genes]


def observed_expression_by_state(adata, genes: list[str]) -> np.ndarray:
    gene_indices = _gene_indices(adata.var_names, genes)
    expression = np.asarray(adata.layers["spliced"][:, gene_indices])
    states = adata.obs["state"].astype(int).to_numpy()
    rows = []
    for state_index in range(num_states_from_adata(adata)):
        mask = states == state_index
        if mask.any():
            rows.append(expression[mask].mean(axis=0))
        else:
            rows.append(np.full(len(genes), np.nan))
    return np.vstack(rows)


def plot_configured_and_observed_expression(
    adata,
    genes: list[str],
    *,
    axes: tuple[plt.Axes, plt.Axes],
    state_names: list[str] | None = None,
) -> None:
    configured_indices = _gene_indices(adata.var_names, genes)
    configured = np.asarray(adata.uns["state_expression"], dtype=float)[:, configured_indices]
    observed = observed_expression_by_state(adata, genes)
    state_indices = np.arange(num_states_from_adata(adata))
    if state_names is None:
        state_names = [str(state) for state in state_indices]
    label_step = max(1, len(genes) // 16)
    tick_positions = np.arange(0, len(genes), label_step)

    panels = [
        (axes[0], configured, "Configured expression", "steady-state target"),
        (axes[1], observed, "Observed expression", "mean raw spliced count"),
    ]
    for ax, matrix, title, colorbar_label in panels:
        im = ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap="viridis")
        ax.set_title(title)
        ax.set_yticks(state_indices)
        ax.set_yticklabels(state_names)
        ax.set_ylabel("State")
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([genes[index] for index in tick_positions], rotation=90, fontsize=7)
        ax.set_xlabel("genes")
        ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=colorbar_label)


def population_over_time(adata) -> pd.DataFrame:
    birth_time = adata.obs["birth_time"].to_numpy(dtype=float)
    times = np.sort(np.unique(birth_time))
    return pd.DataFrame(
        {
            "time": times,
            "cells": [np.count_nonzero(birth_time <= time) for time in times],
        }
    )

## Define the QAD transition and division model

The base graph contains the differentiation routes before feedback is applied. The dynamic transition callback multiplies this base matrix by a Hill factor determined by the number of terminal `D` cells. The proliferation callback assigns a positive division rate to `M2` and uses the same Hill factor to reduce the division rate as `D` cells accumulate. When an `M2` cell divides, both daughters independently choose fates in `Q2`, `A`, or `D`, with zero probability of landing back in `M2`.

In [ ]:
Q1, Q2, A, M1, M2, D = range(6)
NUM_STATES = 6
NUM_GENES = 220
STATE_NAMES = ["Q1", "Q2", "A", "M1", "M2", "D"]
STATE_COLORS = state_palette(NUM_STATES)
STATE_POSITIONS = np.array(
    [
        [-1.4, 0.4],
        [-0.7, 0.4],
        [0.0, 0.0],
        [0.7, 0.4],
        [1.4, 0.4],
        [0.7, -0.55],
    ]
)

base_transition_rates = np.array(
    [
        [0.0, 0.06, 0.0, 0.0, 0.0, 0.0],
        [0.06, 0.0, 0.18, 0.0, 0.0, 0.0],
        [0.0, 0.02, 0.0, 0.16, 0.0, 0.04],
        [0.0, 0.0, 0.0, 0.0, 0.20, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    ]
)

D_HALF_SATURATION = 100.0
HILL_COEFFICIENT = 3.0
M2_DIVISION_RATE = 2.0
DAUGHTER_STATE_PROBS = np.zeros((NUM_STATES, NUM_STATES), dtype=float)
DAUGHTER_STATE_PROBS[M2, [Q2, A, D]] = [0.35, 0.40, 0.25]


def d_feedback_from_count(d_count: int) -> float:
    return 1.0 / (1.0 + (d_count / D_HALF_SATURATION) ** HILL_COEFFICIENT)


def d_feedback(state: SimulationState) -> float:
    d_count = int(np.count_nonzero(state.cell_states == D))
    return d_feedback_from_count(d_count)


def qad_transition_rates(state: SimulationState) -> np.ndarray:
    return base_transition_rates * d_feedback(state)


def m2_proliferation(state: SimulationState) -> tuple[np.ndarray, np.ndarray]:
    division_rates = np.zeros(NUM_STATES, dtype=float)
    division_rates[M2] = M2_DIVISION_RATE * d_feedback(state)

    return division_rates, DAUGHTER_STATE_PROBS


qad_params = SimulationParameters(
    # Latent and population dynamics.
    num_states=NUM_STATES,
    transition_rates=qad_transition_rates,
    proliferation_model=m2_proliferation,

    # State-specific spliced-count targets and U/S dynamics.
    num_genes=NUM_GENES,
    markers_per_state=40,
    marker_overlap={
        (Q1, Q2): 5,
        (Q2, A): 5,
        (A, M1): 5,
        (M1, M2): 5,
        (M2, Q2): 5,
        (M2, A): 5,
        (A, D): 5,
        (M2, D): 5,
    },
    baseline_expression=0.20,
    marker_expression=14.0,
    splicing_rate=.5,
    decay_rate=.5,

    # Initial condition.
    initial_state_probs=[0.4, 0.6, 0.0, 0.0, 0.0, 0.0],

    # Simulation size, time, and reproducibility.
    num_cells=100,
    t_final=200.0,
    dt=0.5,
    rng_seed=42,
)

## Inspect the feedback mechanism

The graph on the left is the base transition system before feedback. The graph on the right shows the same system after many `D` cells have accumulated. Solid grey arrows are ordinary latent transitions. Dashed `M2`-colored arrows show the effective two-daughter output of an `M2` division into `Q2`, `A`, and `D`, with labels equal to `2 * division_rate * fate_probability`. All transition rates and the `M2` division rate use the same feedback multiplier.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
plot_transition_graph(
    base_transition_rates,
    "Base transition + division graph",
    ax=axes[0],
    positions=STATE_POSITIONS,
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
    division_source=M2,
    division_rate=M2_DIVISION_RATE,
    daughter_state_probs=DAUGHTER_STATE_PROBS,
)

suppressed_feedback = d_feedback_from_count(900)
suppressed = base_transition_rates * suppressed_feedback
plot_transition_graph(
    suppressed,
    "With many D cells",
    ax=axes[1],
    positions=STATE_POSITIONS,
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
    division_source=M2,
    division_rate=M2_DIVISION_RATE * suppressed_feedback,
    daughter_state_probs=DAUGHTER_STATE_PROBS,
)

d_counts = np.arange(0, 600)
feedback = [d_feedback_from_count(count) for count in d_counts]
axes[2].plot(d_counts, feedback)
axes[2].set_xlabel("D cells")
axes[2].set_ylabel("transition/division multiplier")
axes[2].set_title("D-dependent freezing")
plt.tight_layout()

## Simulate the QAD system

The final output is AnnData. We add a categorical `state_label` column for readability, while keeping the integer `state` column used by the helper functions.

In [ ]:
qad_adata = simulate_dataset(qad_params)
qad_adata.obs["state_label"] = pd.Categorical(
    [STATE_NAMES[int(state)] for state in qad_adata.obs["state"]],
    categories=STATE_NAMES,
)
qad_adata

In [ ]:
final_d_count = int(np.count_nonzero(qad_adata.obs["state"].astype(int).to_numpy() == D))
pd.Series(
    {
        "initial_cells": qad_params.num_cells,
        "final_cells": qad_adata.n_obs,
        "division_events": int(np.count_nonzero(qad_adata.obs["birth_parent"].to_numpy(dtype=int) >= 0)),
        "final_D_cells": final_d_count,
        "final_feedback_multiplier": d_feedback_from_count(final_d_count),
    }
)

## Inspect the generated data

The population-size curve comes directly from `obs["birth_time"]`: initial cells have birth time `0.0`, and each division appends one second-daughter row with the step-end time at which division occurred. The reused daughter row records the same event in `last_division_time` and increments `generation`.

In [ ]:
qad_adata = preprocess_for_scanpy(qad_adata)
qad_adata

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
curve = population_over_time(qad_adata)
axes[0].step(curve["time"], curve["cells"], where="post")
axes[0].set_xlabel("time")
axes[0].set_ylabel("cells")
axes[0].set_title("M2-driven divisions")

plot_state_counts(
    qad_adata,
    "Final state occupancy",
    ax=axes[1],
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)
plot_scanpy_views(
    qad_adata,
    "Refined QAD",
    axes=axes[2:4],
    palette=STATE_COLORS,
    state_names=STATE_NAMES,
)
plt.tight_layout()

## Configured and observed expression programs

The expression heatmaps show all genes. The configured matrix is the state-by-gene spliced-count target used by the simulator; the observed matrix is the final mean raw spliced count in each state.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
plot_configured_and_observed_expression(
    qad_adata,
    list(qad_adata.var_names),
    axes=axes,
    state_names=STATE_NAMES,
)
plt.tight_layout()